In [ ]:
import sys, subprocess

def _pip(args):
    subprocess.run([sys.executable, "-m", "pip"] + args, check=False)

# Full uninstall — case does not matter to pip
_pip(["uninstall", "-y",
      "rapidocr", "rapidocr_paddle", "rapidocr_onnxruntime", "rapidocr_openvino",
      "onnxruntime", "onnxruntime-gpu", "onnxruntime-training",
      "pillow", "pymupdf", "pdfplumber"])

# Install everything in ONE command, with Pillow explicitly pinned.
# Explicit pin wins over rapidocr's unpinned requirement.
_pip(["install",
      "pillow==10.4.0",        # < 12 → keeps PIL._typing._Ink
      "rapidocr",
      "onnxruntime-gpu",
      "pymupdf",
      "pdfplumber",
      "tqdm"])

print("\n" + "="*60)
print("✅ Install complete.")
print("➡  NOW GO TO:  Run  →  Restart & Clear Cell Outputs")
print("   Then run CELL 2 (verify) and CELL 3 (pipeline).")
print("="*60)

In [1]:
import PIL, onnxruntime as ort, rapidocr

print("Pillow          :", PIL.__version__)
print("onnxruntime     :", ort.__version__)
print("Providers       :", ort.get_available_providers())
print("rapidocr        :", getattr(rapidocr, "__version__", "n/a"))

import torch
print("torch CUDA      :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))

Pillow          : 10.4.0
onnxruntime     : 1.30.0
Providers       : ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
rapidocr        : n/a
torch CUDA      : True
GPU             : Tesla T4


In [3]:
# ══════════════════════════════════════════════════════════════════════════
#  FAST PIPELINE — PyMuPDF native text, RapidOCR fallback only when needed
# ══════════════════════════════════════════════════════════════════════════
import os, io, zipfile, time
from pathlib import Path

import pymupdf
import numpy as np
from PIL import Image
from tqdm import tqdm
from rapidocr import RapidOCR, EngineType
import onnxruntime as ort

# ── Locate source ───────────────────────────────────────────────────────
def _find_source_dir():
    for p in Path("/kaggle/input").rglob("source_pdf"):
        if p.is_dir(): return p
    for p in Path("/kaggle/input").rglob("*"):
        if p.is_dir() and (p / "pdf_list.txt").exists(): return p
    for p in Path("/kaggle/input").rglob("*"):
        if p.is_dir() and any(p.rglob("*.pdf")): return p
    return None

SOURCE_DIR = _find_source_dir()
assert SOURCE_DIR, "No PDF folder found"
print(f"✓ SOURCE_DIR = {SOURCE_DIR}")

OUTPUT_DIR = Path("/kaggle/working/extracted")
ZIP_PATH   = Path("/kaggle/working/extracted_text.zip")
DPI        = 250   # a bit higher than 200 → better OCR on small text

# ── RapidOCR (GPU) ──────────────────────────────────────────────────────
_avail  = ort.get_available_providers()
use_gpu = "CUDAExecutionProvider" in _avail
print(f"→ ORT providers : {_avail}")
print(f"→ Using         : {'GPU 🚀' if use_gpu else 'CPU 🐢'}")

params = {
    "Det.engine_type": EngineType.ONNXRUNTIME,
    "Cls.engine_type": EngineType.ONNXRUNTIME,
    "Rec.engine_type": EngineType.ONNXRUNTIME,
}
if use_gpu:
    params["EngineConfig.onnxruntime.use_cuda"] = True

ocr_engine = RapidOCR(params=params)
_ = ocr_engine(np.zeros((64, 64, 3), dtype=np.uint8))
print("✓ RapidOCR ready\n")

# ── Core extraction ────────────────────────────────────────────────────
#  KEY CHANGE: never call pdfplumber. PyMuPDF does BOTH native text and render.

def _native_text(doc, page_num):
    """Fast native text extraction from a PyMuPDF Document (already open)."""
    try:
        t = doc[page_num].get_text("text")
        return t.strip() if t and t.strip() else None
    except Exception:
        return None

def _ocr_from_doc(doc, page_num, dpi=DPI):
    """Render + OCR one page using an already-open PyMuPDF Document."""
    try:
        pix = doc[page_num].get_pixmap(dpi=dpi)
        img = Image.open(io.BytesIO(pix.tobytes("png")))
        arr = np.array(img)
        if arr.ndim == 3:
            arr = arr[:, :, ::-1]              # RGB → BGR
        res  = ocr_engine(arr)
        txts = getattr(res, "txts", None)
        return "\n".join(txts) if txts else None
    except Exception as e:
        print(f"  ✗ OCR p{page_num}: {e}")
        return None

def process_pdf(pdf_path):
    """Open once with PyMuPDF, extract first/second/last page."""
    try:
        doc = pymupdf.open(str(pdf_path))
        total = len(doc)
        if total == 0:
            doc.close()
            return None

        pages = [0, 1, total - 1] if total >= 2 else [0]
        seen  = set()
        pages = [p for p in pages if not (p in seen or seen.add(p))]

        parts = []
        for p in pages:
            text = _native_text(doc, p)          # ← ~5 ms
            if not text:
                text = _ocr_from_doc(doc, p)     # ← only if native was empty
            if text:
                parts.append(f"--- Page {p + 1} ---\n{text}")

        doc.close()
        return "\n\n".join(parts) if parts else None
    except Exception as e:
        print(f"  ✗ {pdf_path.name}: {e}")
        return None

# ── Collect & process ──────────────────────────────────────────────────
pdf_files = sorted(SOURCE_DIR.rglob("*.pdf"))
print(f"✓ Found {len(pdf_files)} PDF files\n")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
success, failed = 0, []
t0 = time.time()

for pdf_path in tqdm(pdf_files, desc="Processing", unit="pdf"):
    rel     = pdf_path.relative_to(SOURCE_DIR)
    out_pth = (OUTPUT_DIR / rel).with_suffix(".txt")
    out_pth.parent.mkdir(parents=True, exist_ok=True)

    text = process_pdf(pdf_path)
    if text:
        out_pth.write_text(text, encoding="utf-8")
        success += 1
    else:
        failed.append(str(rel))

elapsed = time.time() - t0
print(f"\n{'='*55}")
print(f"DONE  ✓ success={success}   ✗ failed={len(failed)}")
print(f"Time  : {elapsed/60:.1f} min  ({elapsed/max(success,1):.2f} s/pdf)")
if failed:
    print("First 20 failed:")
    for f in failed[:20]:
        print("  -", f)

# ── ZIP + download link ────────────────────────────────────────────────
print("\n→ Creating zip…")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUTPUT_DIR):
        for fname in files:
            fp = os.path.join(root, fname)
            zf.write(fp, os.path.relpath(fp, OUTPUT_DIR))
print(f"✓ Zip: {ZIP_PATH}  ({ZIP_PATH.stat().st_size/1e6:.2f} MB)")

from IPython.display import FileLink, display
display(FileLink(str(ZIP_PATH)))
print("\n👆 Click to download extracted_text.zip")

[INFO] 2026-09-15 07:31:52,782 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-09-15 07:31:52,814 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-09-15 07:31:52,815 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
2026-09-15 07:31:52.833781089 [W:onnxruntime:Default, onnxruntime_pybind_state.cc:855 operator()] No registered plugin EP device found for 'CUDAExecutionProvider' with device_id=0
[WARNING] 2026-09-15 07:31:52,884 [RapidOCR] provider_config.py:90: CUDAExecutionProvider is available, but the inference part is automatically shifted to be executed under CPUExecutionProvider. 
[WARNING] 2026-09-15 07:31:52,885 [RapidOCR] provider_config.py:93: The available lists are ['CPUExecutionProvider']
2026-09-15 07:31:52.834704670 [E:onnxruntime:Default, provider_bridge_ort.cc:2395 TryGetProviderInfo_CUDA] /onnxr

✓ SOURCE_DIR = /kaggle/input/datasets/fayaz042/source-pdf
→ ORT providers : ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
→ Using         : GPU 🚀


[WARNING] 2026-09-15 07:31:52,965 [RapidOCR] provider_config.py:93: The available lists are ['CPUExecutionProvider']
2026-09-15 07:31:52.903164603 [E:onnxruntime:Default, provider_bridge_ort.cc:2395 TryGetProviderInfo_CUDA] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:1988 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library /usr/local/lib/python3.12/dist-packages/onnxruntime/capi/libonnxruntime_providers_cuda.so with error: libcublasLt.so.13: cannot open shared object file: No such file or directory

2026-09-15 07:31:52.903181743 [W:onnxruntime:Default, onnxruntime_pybind_state.cc:1294 CreateExecutionProviderFactoryInstance] Failed to create CUDAExecutionProvider. Require cuDNN 9.* and CUDA 13.*. Please install all dependencies as mentioned in the GPU requirements page (https://onnxruntime.ai/docs/execution-providers/CUDA-ExecutionProvider.html#requirements), make sure they're in the PATH, and that your G

✓ RapidOCR ready

✓ Found 1672 PDF files



Processing:  22%|██▏       | 376/1672 [18:09<3:45:12, 10.43s/pdf][WARNING] 2026-09-15 07:50:12,304 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-09-15 07:50:15,149 [RapidOCR] main.py:132: The text detection result is empty
Processing:  97%|█████████▋| 1629/1672 [1:45:09<03:02,  4.24s/pdf][WARNING] 2026-09-15 09:17:11,621 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-09-15 09:17:14,258 [RapidOCR] main.py:132: The text detection result is empty
Processing: 100%|██████████| 1672/1672 [1:48:52<00:00,  3.91s/pdf]



DONE  ✓ success=1671   ✗ failed=1
Time  : 108.9 min  (3.91 s/pdf)
First 20 failed:
  - public_notices/7091_NDLS_portal_Public_notice-31-03-2021.pdf

→ Creating zip…
✓ Zip: /kaggle/working/extracted_text.zip  (3.27 MB)


/kaggle/working/extracted_text.zip


👆 Click to download extracted_text.zip


In [4]:
# ══════════════════════════════════════════════════════════════════════════
#  ZIP the extracted folder and place it in Kaggle's output directory
#  (/kaggle/working/  ←  this is what the "Output" tab shows)
# ══════════════════════════════════════════════════════════════════════════
import os, zipfile, shutil
from pathlib import Path

EXTRACTED_DIR = Path("/kaggle/working/extracted")
ZIP_PATH      = Path("/kaggle/working/extracted_text.zip")

# ── Safety checks ──────────────────────────────────────────────────────
assert EXTRACTED_DIR.exists(), f"❌ {EXTRACTED_DIR} does not exist. Run the extraction cell first."
assert any(EXTRACTED_DIR.rglob("*.txt")), f"❌ No .txt files inside {EXTRACTED_DIR}."

# ── Remove any previous zip so we start clean ─────────────────────────
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

# ── Count files first ──────────────────────────────────────────────────
txt_files = list(EXTRACTED_DIR.rglob("*.txt"))
print(f"→ Found {len(txt_files)} .txt files to archive")

# ── Create ZIP with progress ───────────────────────────────────────────
print(f"→ Writing {ZIP_PATH} …")
with zipfile.ZipFile(ZIP_PATH, "w",
                    compression=zipfile.ZIP_DEFLATED,
                    compresslevel=6) as zf:
    for i, fp in enumerate(txt_files, 1):
        arcname = fp.relative_to(EXTRACTED_DIR)
        zf.write(fp, arcname)
        if i % 200 == 0 or i == len(txt_files):
            print(f"   {i}/{len(txt_files)} …")

# ── Verify the archive is readable ─────────────────────────────────────
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    bad = zf.testzip()
    n_in_zip = len(zf.namelist())
assert bad is None, f"❌ Corrupt entry in zip: {bad}"
print(f"✓ Zip verified: {n_in_zip} entries, no corruption")

# ── Sizes ──────────────────────────────────────────────────────────────
extracted_mb = sum(f.stat().st_size for f in txt_files) / 1e6
zip_mb       = ZIP_PATH.stat().st_size / 1e6
print(f"\n  extracted/  total : {extracted_mb:.2f} MB")
print(f"  extracted_text.zip: {zip_mb:.2f} MB  "
      f"(compression {100*(1 - zip_mb/max(extracted_mb,1e-9)):.0f}%)")

# ── List Kaggle output folder ──────────────────────────────────────────
print("\n→ Contents of /kaggle/working/ (this is your Kaggle Output folder):")
for p in sorted(Path("/kaggle/working").iterdir()):
    if p.is_file():
        print(f"   {p.name:35s} {p.stat().st_size/1e6:8.2f} MB")
    else:
        n = sum(1 for _ in p.rglob("*") if _.is_file())
        print(f"   {p.name + '/':35s} {n:>6} files")

print(f"\n✅ DONE — download {ZIP_PATH.name} from the notebook's")
print("   right sidebar → 'Output' tab → Download.")

→ Found 1671 .txt files to archive
→ Writing /kaggle/working/extracted_text.zip …
   200/1671 …
   400/1671 …
   600/1671 …
   800/1671 …
   1000/1671 …
   1200/1671 …
   1400/1671 …
   1600/1671 …
   1671/1671 …
✓ Zip verified: 1671 entries, no corruption

  extracted/  total : 7.16 MB
  extracted_text.zip: 3.27 MB  (compression 54%)

→ Contents of /kaggle/working/ (this is your Kaggle Output folder):
   .virtual_documents/                      1 files
   extracted/                            1671 files
   extracted_text.zip                      3.27 MB

✅ DONE — download extracted_text.zip from the notebook's
   right sidebar → 'Output' tab → Download.
